# Do unlearned facts keep their circuits? Colab runner
This notebook only runs the scripts of the repository. **Runtime > Change runtime type > GPU (T4)**, then run the cells in order.
Upload `unlearning_circuits_repo.zip` (the repository) to the session first, or clone it from GitHub in the first cell.

In [ ]:
# 1. get the code (zip uploaded to the session; or: !git clone <your repo url> repo)
!rm -rf repo && mkdir repo && unzip -q -o unlearning_circuits_repo.zip -d repo
%cd repo

In [ ]:
# 2. install (transformer_lens must stay on 2.x)
!pip -q install einops jaxtyping beartype better-abc fancy-einsum typeguard transformers datasets wandb pandas scipy matplotlib rich sentencepiece accelerate
!pip -q install --no-deps transformer_lens==2.16.1

In [ ]:
# 3. data (CounterFact, 43 MB)
!bash data/download_counterfact.sh

In [ ]:
# 4. optional: keep results on Google Drive so a disconnect does not lose them
from google.colab import drive
drive.mount("/content/drive")
RESULTS = "/content/drive/MyDrive/unlearning_circuits_results"   # or "results" to keep them on the session only

## 5. Run
Every stage is resumable: finished (stage, learning rate, seed) results are skipped, so after a disconnect just run the cell again.
Your earlier results in `RESULTS` are kept. Run these cells one at a time; the long ones take a few hours each on a T4.
1. Fast: the heavy-stage method comparison comes from runs that already exist (no new training).
2. `surface` is quick (Phase 2 and 3 checks). 3. `ladder`, 4. `methods`, 5. `locate` are the long ones (all three learning rates; `methods` also runs the p<1e-4 rung).
`locate` now also measures the heavy stage; older `locate` results (light only) are skipped unless you pass `--force` (cell 5b).


In [ ]:
!python run_experiments.py --config configs/main.json --results {RESULTS} --stages methods analyze   # (1) mostly skips what exists

In [ ]:
!python run_experiments.py --config configs/main.json --results {RESULTS} --stages surface analyze   # (2)

In [ ]:
!python run_experiments.py --config configs/main.json --results {RESULTS} --stages ladder analyze     # (3) ladder at the other learning rates

In [ ]:
!python run_experiments.py --config configs/main.json --results {RESULTS} --stages methods analyze    # (4) methods at all learning rates + deeper rung

In [ ]:
!python run_experiments.py --config configs/main.json --results {RESULTS} --stages locate analyze --force   # (5b) restoration tests with heavy stage, all learning rates (recomputes old locate results)

In [ ]:
!python run_experiments.py --config configs/main.json --results {RESULTS} --stages recovery swap-locate swap-survival analyze   # only needed if not run before

## 6. Reports and figures

In [ ]:
import os, glob
from IPython.display import Markdown, Image, display
reports = ["analysis_report.md", "phase_checks.md", "strength_report.md", "methods_report.md", "methods_report_heavy.md",
           "methods_report_p1e-4.md", "methods_report_p1e-4_heavy.md", "methods_report_lr1e-06.md", "methods_report_lr1e-05.md",
           "recovery/lr3e-06/recovery_report.md", "swap_comparison.md"]
reports += [os.path.relpath(p, RESULTS) for p in sorted(glob.glob(os.path.join(RESULTS, "locate", "lr*", "locate_report.md")))]
for f in reports:
    p = os.path.join(RESULTS, f)
    if os.path.exists(p):
        display(Markdown(open(p).read()))
for f in ["fig_survival.png", "fig_robustness.png", "fig_strength.png", "recovery/lr3e-06/fig_recovery.png", "locate/lr3e-06/fig_locate_light.png"]:
    p = os.path.join(RESULTS, f)
    if os.path.exists(p):
        display(Image(p))


In [ ]:
# 7. one zip with everything, for download
!cd {RESULTS} && zip -q -r /content/results_bundle.zip . -x "*/log_seed*" && ls -la /content/results_bundle.zip
from google.colab import files
files.download("/content/results_bundle.zip")